# Precision, Recall, and Threshold

In this worksheet you will apply precision-recall analysis to the [Wine Recognition dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset), a built-in sklearn dataset. You will binarize the multi-class target, build and tune a logistic regression classifier, and analyze the precision-recall trade-off by sweeping decision thresholds.

***Summary***
1. [Load and Explore the Data](#load-data)
2. [Precision and Recall from Scratch](#precision-recall-scratch)
3. [Build and Tune a Logistic Regression Model](#build-tune)
4. [Threshold Analysis](#threshold-analysis)
5. [Apply to a New Domain](#new-domain)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

***
<a id='load-data'></a>
## 1. Load and Explore the Data

Load the Wine Recognition dataset from sklearn and binarize the target to a two-class problem: class 0 (high-quality wine type) as the positive class (label = 1), and classes 1 and 2 combined as the negative class (label = 0). This binarization gives a moderately imbalanced dataset suitable for precision-recall analysis.

**Q1a) Load the wine dataset. Store the feature matrix in `X` and the binarized label vector in `y` (1 where the original class equals 0, else 0).**

*Hint:* Use [`sklearn.datasets.load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)

In [ ]:
from sklearn.datasets import load_wine

### YOUR CODE HERE ###
data = load_wine()
X = ...
y = ...

#print('X shape:', X.shape)
#print('Label counts - 0:', (y == 0).sum(), '| 1:', (y == 1).sum())

**Q1b) Print the feature names for this dataset. Then compute and print the mean of the first feature for each class (positive vs. negative).**

*Hint:* `data.feature_names` gives the list of feature names.

*Hint:* Use boolean indexing: `X[y == 1, 0]` selects the first feature for positive-class samples.

Store the two means in `mean_positive` and `mean_negative`.

Expected output: two float values printed to the console.

In [ ]:
### YOUR CODE HERE ###
mean_positive = ...
mean_negative = ...

#print('Feature:', data.feature_names[0])
#print(f'Mean (class 0 - positive): {mean_positive:.4f}')
#print(f'Mean (class 1/2 - negative): {mean_negative:.4f}')

***
<a id='precision-recall-scratch'></a>
## 2. Precision and Recall from Scratch

Before using sklearn's metrics, implement precision and recall directly from a confusion matrix. This ensures you understand what the numbers mean before relying on library functions.

**Q2a) Split the data into an 80/20 train/test set with `random_state=42`. Then define a "dummy" classifier that always predicts the majority class (0). Compute its confusion matrix on the test set. Store it in `C_dummy`.**

*Hint:* `np.zeros_like(y_test)` creates an array of zeros with the same shape as `y_test`.

*Hint:* Use [`sklearn.metrics.confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

In [ ]:
from sklearn.model_selection import train_test_split

random_state = 42  # don't change these lines
test_size = 0.2    # don't change these lines

### YOUR CODE HERE ###
X_train, X_test, y_train, y_test = ...
y_pred_dummy = ...
C_dummy = ...

#print('Dummy classifier confusion matrix:')
#print(C_dummy)

**Q2b) Using `C_dummy`, compute precision and recall for the dummy classifier WITHOUT using `sklearn.metrics.precision_score` or `sklearn.metrics.recall_score`. Store results in `dummy_precision` and `dummy_recall`.**

Expected output: precision = 0.0 (or undefined), recall = 0.0.

*Hint:* Handle the division-by-zero case when TP + FP = 0 by returning 0.0 for precision.

In [ ]:
### YOUR CODE HERE ###
tp = C_dummy[1, 1]
fp = C_dummy[0, 1]
fn = C_dummy[1, 0]

dummy_precision = ...
dummy_recall = ...

#print(f'Dummy precision: {dummy_precision:.3f}')
#print(f'Dummy recall:    {dummy_recall:.3f}')

*ANSWER HERE*

Why does the dummy classifier have the precision and recall values you computed? What does this tell you about using accuracy as a metric on imbalanced datasets?

***
<a id='build-tune'></a>
## 3. Build and Tune a Logistic Regression Model

Build a scikit-learn Pipeline and tune the regularization parameter C via cross-validation. Run the search twice - once optimizing for recall, once for precision.

**Q3a) Build a pipeline with `StandardScaler` and `LogisticRegression` (use `max_iter=2000`). Tune the regularization parameter `C` over `np.logspace(-3, 3, 10)` using 4-fold `StratifiedKFold` with `scoring='recall'`. Store the best pipeline in `pipeline_best_recall`.**

*Hint:* Use [`sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) with `cv=split` where `split = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

split = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)  # don't change these lines
parameters = {'lr__C': np.logspace(-3, 3, 10)}                     # don't change these lines

### YOUR CODE HERE ###
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=2000))
])

pipeline_best_recall = ...

#print('Best C (recall-optimized):', pipeline_best_recall.named_steps['lr'].C)

**Q3b) Compute the cross-validation recall and precision for `pipeline_best_recall` on the training set. Store them in `cv_recall` and `cv_precision`.**

Expected output: two floats between 0 and 1.

In [ ]:
### YOUR CODE HERE ###
cv_recall = ...
cv_precision = ...

#print(f'CV Recall:    {cv_recall:.3f}')
#print(f'CV Precision: {cv_precision:.3f}')

**Q3c) Repeat the GridSearchCV with `scoring='precision'`. Store the best pipeline in `pipeline_best_precision`. Then compare the C values chosen by each search.**

*Hint:* Use the same `parameters` and `split` variables as above.

In [ ]:
### YOUR CODE HERE ###
pipeline_best_precision = ...

#print(f'C (recall-optimized):    {pipeline_best_recall.named_steps["lr"].C:.4f}')
#print(f'C (precision-optimized): {pipeline_best_precision.named_steps["lr"].C:.4f}')

***
<a id='threshold-analysis'></a>
## 4. Threshold Analysis

Sweep the decision threshold and observe how precision and recall trade off. Use the recall-optimized pipeline from Q3a.

**Q4a) Fit `pipeline_best_recall` on `X_train`, then sweep 20 decision thresholds between 0.05 and 0.95. At each threshold, compute precision and recall on `X_test`. Store results in arrays `precisions` and `recalls`. Then plot both curves against the threshold values on the same axes.**

*Hint:* Use `pipeline_best_recall.predict_proba(X_test)[:, 1]` to get probability scores.

*Hint:* Use `confusion_matrix(y_test, (probs >= t).astype(int))` at each threshold `t`.

Expected output: a matplotlib figure with two curves (precision in red, recall in blue).

`# TODO: capture expected plot output as assets/q4a_expected.PNG`

In [ ]:
### YOUR CODE HERE ###
pipeline_best_recall.fit(X_train, y_train)

n = 20
thresh = np.linspace(0.05, 0.95, n)
precisions = np.zeros(n)
recalls = np.zeros(n)

for i, t in enumerate(thresh):
    ...

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
### YOUR CODE HERE ###
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall vs. Decision Threshold (Wine Dataset)')
ax.legend()
plt.tight_layout()
plt.show()

**Q4b) Find the decision threshold from your sweep at which precision and recall are closest to equal (minimum absolute difference). Store that threshold in `balanced_threshold` and the precision and recall at that point in `balanced_precision` and `balanced_recall`.**

Expected output: three float values printed to the console.

In [ ]:
### YOUR CODE HERE ###
balanced_threshold = ...
balanced_precision = ...
balanced_recall = ...

#print(f'Balanced threshold: {balanced_threshold:.2f}')
#print(f'Precision at balance: {balanced_precision:.3f}')
#print(f'Recall at balance:    {balanced_recall:.3f}')

***
<a id='new-domain'></a>
## 5. Apply to a New Domain

Apply the same logistic regression precision-recall framework to a different binary classification problem: detecting whether a breast tissue sample is malignant, using sklearn's built-in Breast Cancer Wisconsin dataset. This confirms you can transfer the workflow without modifying the dataset-loading steps.

**Q5a) Load the built-in `load_breast_cancer` dataset. Store features in `X_bc` and labels in `y_bc` (label 0 = malignant in the sklearn encoding, label 1 = benign - to make malignant the positive class, set `y_bc = 1 - data_bc.target`). Split into 80/20 train/test with `random_state=42`. Store in `X_bc_train`, `X_bc_test`, `y_bc_train`, `y_bc_test`.**

*Hint:* Use [`sklearn.datasets.load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)

In [ ]:
from sklearn.datasets import load_breast_cancer

### YOUR CODE HERE ###
data_bc = load_breast_cancer()
X_bc = ...
y_bc = ...  # 1 = malignant (the positive class we want to detect)

X_bc_train, X_bc_test, y_bc_train, y_bc_test = ...

#print('X_bc shape:', X_bc.shape)
#print('Malignant count (y=1):', (y_bc == 1).sum())

**Q5b) Reuse the same `parameters` and `split` from Section 3. Run `GridSearchCV` with `scoring='recall'` on the breast cancer training data. Store the best pipeline in `pipeline_bc_recall`. Compute and print the test-set recall and precision at the default threshold (0.5). Store them in `bc_recall` and `bc_precision`.**

*Hint:* Use `pipeline_bc_recall.predict(X_bc_test)` for predictions at the default threshold.

In [ ]:
### YOUR CODE HERE ###
pipeline_bc_recall = ...
pipeline_bc_recall.fit(X_bc_train, y_bc_train)

y_pred_bc = pipeline_bc_recall.predict(X_bc_test)
C_bc = confusion_matrix(y_bc_test, y_pred_bc)
tp, fp, fn = C_bc[1, 1], C_bc[0, 1], C_bc[1, 0]

bc_recall = ...
bc_precision = ...

#print(f'Breast cancer test recall:    {bc_recall:.3f}')
#print(f'Breast cancer test precision: {bc_precision:.3f}')

*ANSWER HERE*

Compare `bc_recall` to the recall you achieved on the wine dataset. Which dataset was easier for the logistic regression classifier to achieve high recall on? Propose one reason why based on the dataset descriptions.